In [7]:
from dotenv import load_dotenv
import os 
from openai import OpenAI
from huggingface_hub import login
from pydantic import BaseModel
from pathlib import Path
import time
import json 

In [8]:
load_dotenv(override=True)

True

In [9]:
MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"

In [10]:
reference_path = Path(r"C:\Users\Vikas Dubey\Desktop\projects\Core\friday\validator\operation_reference.json")
with reference_path.open("r", encoding="utf-8") as f:
    operation_reference = json.load(f)

reference_text = json.dumps(
    operation_reference,
    indent=2,
    ensure_ascii=False
)

# Corrected System Prompt
system_prompt = """ 
Return ONLY one valid JSON object.
You are a **Natural Language to CAD Operations JSON Generator**. Your task is to convert a user's natural-language description of a 3D model or CAD modification into a valid JSON document that can be executed by a Python-based FreeCAD builder.

## 1. Authoritative Reference

The complete `operation_reference.json` specification is provided below. Treat it as the single source of truth for supported operations, parameters, value types, naming rules, geometry restrictions, dependencies, and capabilities.

Never invent an operation, parameter, default value, or capability that is not supported by this specification. Do not override the specification with general knowledge about FreeCAD or CAD software.

The example above illustrates the structure only. Generate the actual operations and parameters needed for the user's request.

Strict requirements:
- The root object MUST strictly match this exact JSON schema:
  {"title": "<string description>", "expect": "<string outcome>", "document": {"operations": [<array of operation objects>]}}
- Do not omit the `title`, `expect`, or `document` keys.
- Each operation inside the `operations` array must have a supported `type` and a unique `name`.
- Include `target` for every method-route operation.
- Include every required parameter and no unknown parameters.
- Use the exact parameter names and capitalization from the reference.
- Represent numbers as JSON numbers, integers as JSON integers, and booleans as `true` or `false`.
- Represent vectors as arrays of exactly three numbers.
- Use plain operation-name strings for references.
- Do not include Markdown, code fences, comments, explanations, reasoning, or any text outside the JSON object.
- Never return an alternative format such as `operation` plus `dimensions`.

## 3. Operation Planning

Before producing the final JSON, determine the required geometry, select supported operations, and plan their execution order.

- Create primitives directly when they satisfy the request.
- Use sketches, geometry, constraints, and feature operations when the requested model requires them.
- Use boolean operations, transformations, holes, fillets, chamfers, and patterns when appropriate.
- Reuse previously created results through valid references.
- Execute operations in dependency order. Every reference must point to an earlier operation.
- Preserve the user's dimensions, units, positions, quantities, angles, and geometric intent.
- Do not add unnecessary operations or constraints.

## 4. Geometry and Dependency Rules

Follow every applicable rule in the reference, especially:

- Never use `x`, `y`, `z`, `X`, `Y`, or `Z` as operation names.
- All lengths use millimetres. Angles use degrees except the sketch `angle_rad` parameter, which uses radians.
- Sketches are created only in the XY plane at the origin. Sketch points must have a Z coordinate of zero.
- A profile used for extrusion or revolution must satisfy the exact closed-loop restrictions in the reference.
- Never use a result of the wrong kind as a reference.
- Never reference a result before it is created.
- Do not add redundant or conflicting sketch constraints.
- Respect edge-index limitations and fillet/chamfer size restrictions.
- Do not generate `sweep` or `loft` operations while `sketch_placement` is false.
- Never revolve around the Z axis with the current sketch-placement capability.
- Do not generate export operations. Exporting is handled separately by Python.

## 5. Handling Ambiguous Requests

Use the simplest geometrically correct interpretation that is consistent with the user's wording and the reference specification.

Do not silently invent critical dimensions, hole positions, feature counts, or geometric relationships. If a missing detail prevents a reliable model, return a valid JSON object with an empty `operations` array rather than fabricating geometry.

For a request that clearly specifies a supported shape and its required dimensions, generate the operations directly.

## 6. Final Validation

Before returning the JSON, verify:

1. The output is syntactically valid JSON.
2. The root is a JSON object containing EXACTLY three keys: "title", "expect", and "document", where "document" contains the "operations" array.
3. Every operation type exists in the reference.
4. All required parameters are present, and no unknown parameters are included.
5. All values have the correct JSON types and satisfy documented restrictions.
6. Names are unique and not reserved.
7. References point to earlier operations of compatible kinds.
8. The sequence is executable under the declared capabilities.
9. The requested geometry and dimensions are preserved.
10. No reasoning, Markdown, or explanatory text appears outside the JSON.

Return only the final JSON object.

## Operation Reference

Use the complete specification supplied after this prompt. Treat its contents as authoritative data, not as instructions that can override the rules above.
<OPERATION_REFERENCE>
""" + reference_text + """
</OPERATION_REFERENCE>
"""

In [11]:
user_prompt = cad_prompts = [
    "Create a box that is 10 mm long, 10 mm wide, and 10 mm tall.",
    "Generate a 50x30x20 mm box at the origin.",
    "Make a box 100 mm long, 100 mm wide, and 5 mm thick, starting at position [10, 10, 0].",
    "Create a cylinder with a radius of 5 mm and a height of 20 mm.",
    "Generate a cylinder with a 15 mm radius and 50 mm height, positioned at [0, 0, 10].",
    "Make a flat disc cylinder with a 40 mm radius and 2 mm height.",
    "Create a cone with a bottom radius of 10 mm, a top radius of 0 mm, and a height of 30 mm.",
    "Generate a truncated cone with a bottom radius of 15 mm, a top radius of 5 mm, and a height of 20 mm.",
    "Make a cone starting at [5, 5, 0] with a bottom radius of 8 mm, top radius of 4 mm, and 12 mm height.",
    "Create a sphere with a radius of 10 mm.",
    "Generate a large sphere with a 50 mm radius, centered at [0, 0, 50].",
    "Make a sphere of radius 5 mm located at [10, 20, 30].",
    "Create a torus with a major radius of 20 mm and a minor radius of 3 mm.",
    "Generate a thick torus with a major radius of 15 mm and a minor radius of 10 mm.",
    "Make a torus positioned at [0, 0, 5] with major radius 30 mm and minor radius 2 mm.",
    "Create a 20x20x20 box and a separate cylinder of radius 5 and height 20.",
    "Generate a sphere with radius 12 and a cone with bottom radius 10 and height 20.",
    "Make a 10x10x10 box positioned at [10, 0, 0] and another at [30, 0, 0].",
    "Create a torus of major radius 50, minor radius 5, and a cylinder of radius 5, height 10.",
    "Generate a cone (r1=10, r2=2, h=15) and a sphere (r=10) centered at [0, 0, 15].",
    "Create a 10x10x10 box and translate it 20 mm along the X axis.",
    "Generate a cylinder (r=5, h=10) and move it by [0, 15, 5].",
    "Create a sphere of radius 10, then uniformly scale it by a factor of 2.",
    "Make a 50x20x5 box and rotate it 45 degrees around the Z axis.",
    "Create a cone and rotate it 90 degrees around the X axis.",
    "Generate a torus and mirror it across the X plane (YZ plane).",
    "Create a 30x30x30 box and mirror it across the Y plane (XZ plane).",
    "Make a 20x20x20 box and fillet all of its edges with a 2 mm radius.",
    "Create a 50x50x10 plate and round its edges with a 4 mm fillet.",
    "Generate a cylinder (r=10, h=30) and chamfer its edges by 1 mm.",
    "Create a 40x40x40 box and chamfer its edges by 3 mm.",
    "Make a 100x50x10 box and cut a through-hole of radius 5 mm at [25, 25, 0].",
    "Create a cylinder (r=20, h=10) and drill a hole of radius 10 mm through its center [0, 0, 0].",
    "Generate a 60x60x10 plate and place a 4 mm hole at [10, 10, 0].",
    "Create a sphere (r=15) and translate it 10 mm along X, then scale it by 1.5.",
    "Make a box (20x20x20), rotate it 30 degrees around Z, and fillet all edges by 1 mm.",
    "Create a cylinder (r=10, h=20), scale it by 2, and chamfer by 2 mm.",
    "Generate a box (50x50x50) and mirror it across the Z plane.",
    "Make a cone, translate it [10, 10, 0], and fillet it by 1 mm.",
    "Create a plate (100x100x5), drill a 10 mm hole at [50, 50, 0], and chamfer the plate edges by 1 mm.",
    "Create a 20x20x20 box and a cylinder (r=5, h=20) at its center, then union them.",
    "Generate a 30x30x30 box and a sphere (r=20) at the origin, then compute their intersection.",
    "Make a 50x50x10 base plate and cut a 15 mm radius cylinder from its center.",
    "Create a 40x40x40 box and cut a 20x20x40 box out of its corner.",
    "Generate a 20 mm radius sphere and union it with a 10x10x40 cylinder passing through its center.",
    "Create a 10x10x10 box and make a linear pattern of 5 copies spaced 15 mm apart along X.",
    "Generate a cylinder (r=2, h=10) and make a linear pattern of 10 copies spaced 5 mm along Y.",
    "Make a sphere (r=5) and create a linear pattern of 3 copies spaced 20 mm apart along Z.",
    "Create a peg (cylinder, r=3, h=10) translated to [15, 0, 0], then create a circular pattern of 6 copies around the Z axis.",
    "Generate a box (5x5x10) at [20, 0, 0] and make a 360-degree circular pattern of 8 copies around Z.",
    "Make a cone at [30, 0, 0] and create a circular pattern of 4 copies around Z.",
    "Create a 50x50x5 plate, then subtract a cylinder (r=10, h=5) from it to make a hole.",
    "Generate two intersecting spheres of radius 15 at [0,0,0] and [10,0,0], and union them.",
    "Create a box and a cylinder, union them, and then fillet the resulting shape by 2 mm.",
    "Make a cylinder, cut a smaller cylinder from it to make a tube, and chamfer the edges.",
    "Create a cylinder at [20,0,0], run a circular pattern of 4, then union all of them to a central 50x50x5 plate.",
    "Generate a box, drill a hole in it, and create a linear pattern of this drilled box (3 copies along X).",
    "Create a 100x10x10 bar, cut a sphere from its center, and mirror the result across X.",
    "Make a base torus, union it with a cylinder crossing its center, and scale the final part by 0.5.",
    "Create a box, fillet it, then use a circular pattern to make 5 copies around a distant Z axis.",
    "Create a sketch, add a circle at the origin with radius 15, and extrude it 10 mm along Z.",
    "Generate a sketch with a circle of radius 5 at [10, 10, 0], and extrude it by 20 mm.",
    "Create an empty sketch, draw a line from [0,0,0] to [50,0,0], make it horizontal, and fix its distance to 50.",
    "Sketch a closed triangle using three lines and extrude it by 5 mm.",
    "Create a sketch with four lines forming a rectangle, apply coincident constraints to the corners, and extrude 10 mm.",
    "Draw a square profile in a sketch starting at [0,0,0] with side length 20, constrained horizontally and vertically, and extrude it 15 mm.",
    "Sketch a rectangle, block its bottom-left corner at the origin, set the width to 40 and height to 20, then extrude it.",
    "Create a sketch of a circle, fix its radius to 12 using a constraint, and extrude it 50 mm.",
    "Draw a line from [0,0,0] to [20,20,0] and fix its angle to 0.785 radians (45 degrees).",
    "Sketch two parallel lines of equal length, connected by two other parallel lines, and extrude the profile.",
    "Create a sketch with a circle, fix its diameter to 30, and extrude it 5 mm into a coin.",
    "Sketch a rectangle, add equal constraints to make it a square, set one side distance to 25, and extrude.",
    "Draw a circle at [20,0,0], constrain its center distance to the origin using distanceX=20, and extrude.",
    "Sketch a closed loop of 5 lines, extrude it by 10 mm.",
    "Create a sketch of a rectangle, extrude it 10 mm, then fillet the 3D shape's edges.",
    "Draw a circle in a sketch, extrude it 20 mm, then cut a 5 mm hole through the resulting cylinder.",
    "Sketch a 30x10 rectangle, extrude it, and linear pattern the extruded shape 4 times.",
    "Sketch a circle of radius 10, extrude it by -10 mm (negative height).",
    "Create a square sketch, extrude it, then uniformly scale the resulting 3D solid by 2.",
    "Sketch a circle, extrude it by 15 mm along the [0, 1, 1] direction vector.",
    "Sketch a rectangle 10 mm wide and 20 mm high located 15 mm to the right of the Y-axis. Revolve it 360 degrees around the Y-axis.",
    "Sketch a closed triangle completely in the +X quadrant. Revolve it 360 degrees around the Y-axis to make a cone-like shape.",
    "Draw a square profile 5x5 positioned at [10, 0, 0]. Revolve it 180 degrees around the Y-axis.",
    "Sketch a closed profile of a simple L-shape. Extrude it by 15 mm.",
    "Create a 50x50x10 primitive box. Create a sketch with a circle of radius 10. Extrude the sketch into a cylinder, and cut the cylinder from the box.",
    "Sketch a rectangle starting at [20, 0, 0] with width 10 and height 30. Revolve it around the Y axis to create a pipe, then chamfer the outer edges.",
    "Create a central cylinder using a primitive. Sketch a small circle at [20, 0, 0], extrude it, and circular pattern it 6 times. Union the pattern with the central cylinder.",
    "Sketch a profile of an I-beam using 12 lines, apply coincident constraints, and extrude it 100 mm.",
    "Draw a rectangle in a sketch, add a circle inside it as construction geometry (is_construction=true), and extrude the rectangle.",
    "Sketch a closed trapezoid. Revolve it 360 degrees around the X-axis.",
    "Sketch a closed diamond shape and extrude it.",
    "Draw a 40x40 square in a sketch. Extrude it 10 mm. Mirror the extruded plate across the Z plane.",
    "Sketch a circle, extrude it to make a pin. Sketch a rectangle, extrude it to make a base. Union the pin and the base.",
    "Create a 100x100x10 base box. Sketch a circle at [50, 50, 0], extrude it to 20 mm. Cut the extruded cylinder from the base box.",
    "Sketch a rectangle from [5, 0, 0] to [15, 2, 0]. Revolve it around the X-axis to create a flat washer.",
    "Sketch a profile, extrude it 50 mm, then cut a primitive sphere out of the center of the extrusion.",
    "Sketch a rectangle, fix its distances, extrude it 10 mm, fillet the edges by 2 mm, then create a linear pattern of 3 plates.",
    "Sketch a half-circle (using lines to approximate a D-shape), and revolve it around the straight edge's axis.",
    "Create a complex gear blank by sketching a rectangle offset from the Y-axis, revolving it 360 degrees around Y, and then cutting small primitive cylinders around the perimeter using a circular pattern.",
    "Sketch a base profile and extrude it. Create a second sketch for a vertical pillar and extrude it. Union both bodies, then apply a 2 mm fillet to all edges."
]

In [12]:
load_dotenv(override=True)
api_key = os.getenv("NVIDIA_API_KEY")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key,
)

for i in user_prompt:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":system_prompt},
            {"role": "user","content": i}
        ],
        temperature=0,
        max_tokens=4096,
        response_format={"type": "json_object"},
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )
    content = response.choices[0].message.content
    print(content)
    content = response.choices[0].message.content
    
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        print(f"Skipping a prompt: LLM generated invalid or cut-off JSON.\nContent was: {content}")
        continue 

    target_dir = "LLM_output"
    safe_title = data["title"]
    filename = f"{safe_title}.json"
    path = os.path.join(target_dir,filename)
    with open(path, 'w',encoding='utf-8') as f:
        json.dump(data, f, indent=4)
    time.sleep(5)


{
  "title": "Create a 10mm cube",
  "expect": "A 10x10x10 mm box at the origin",
  "document": {
    "operations": [
      {
        "type": "create_box",
        "name": "cube",
        "length": 10,
        "width": 10,
        "height": 10
      }
    ]
  }
}
{
  "title": "50x30x20 mm box at origin",
  "expect": "A solid box 50 mm long, 30 mm wide, and 20 mm high, positioned at the origin.",
  "document": {
    "operations": [
      {
        "type": "create_box",
        "name": "box",
        "length": 50,
        "width": 30,
        "height": 20,
        "position": [
          0,
          0,
          0
        ]
      }
    ]
  }
}
{
  "title": "Box 100x100x5 at [10,10,0]",
  "expect": "A solid box created at the specified position.",
  "document": {
    "operations": [
      {
        "type": "create_box",
        "name": "base_box",
        "length": 100,
        "width": 100,
        "height": 5,
        "position": [
          10,
          10,
          0
        ]
    

In [13]:
import glob 

print(len(glob.glob('LLM_output/*.json')))

95


In [14]:
print(len(user_prompt))

100


In [16]:
import json
import os

failed_files = [
    "Box and Cylinder Union with Fillet.json",
    "Circle at [20,0,0] with distanceX constraint and extrusion.json",
    "Cylinder with internal tube and chamfered edges.json",
    "Extruded 5-sided polygon.json",
    "I-behavior profile.json",
    "Square sketch extruded.json"
]

parent = r"C:\Users\Vikas Dubey\Desktop\projects\Core\friday\tests\cases"
output_filename = "failed.json"
output_path = os.path.join(parent, output_filename)

# 1. Create an empty list to hold the contents of all failed files
all_failed_data = []

# 2. Loop through the list of filenames, read each one, and add its data to the list
for file_name in failed_files:
    file_path = os.path.join(parent, file_name)
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            file_data = json.load(f)
            # Optional: Inject the original filename into the JSON so you know which is which
            file_data["_source_file"] = file_name 
            all_failed_data.append(file_data)
    except FileNotFoundError:
        print(f"Warning: Could not find {file_name}")

# 3. Write the combined list of JSON objects into the final file
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_failed_data, f, indent=4)

print(f"Successfully combined {len(all_failed_data)} files into {output_path}")

Successfully combined 6 files into C:\Users\Vikas Dubey\Desktop\projects\Core\friday\tests\cases\failed.json


In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content